### Setup Environment

In [ ]:
!git clone https://github.com/NhuGiap04/Fk-Diffusion-Steering.git

In [ ]:
%cd Fk-Diffusion-Steering
!git pull

In [ ]:
%pip install -r requirements.txt

### Feynman-Kac Steering

In [ ]:
%cd text_to_image

import os
# os.environ["CUDA_VISIBLE_DEVICES"] = ''
# os.environ['HF_HOME'] = ''

import json
from PIL import Image
import matplotlib.pyplot as plt
import argparse
from datetime import datetime
import numpy as np
import sys
sys.path.append('fkd_diffusers')

import torch
from diffusers import DDIMScheduler

from launch_eval_runs import do_eval

In [ ]:
# Set args
"""
model_choices:

stable-diffusion-xl
stable-diffusion-v1-5
stable-diffusion-v1-4
stable-diffusion-2-1
"""

args = dict(
    seed=0,
    output_dir="output",
    eta=1.0,
    metrics_to_compute="ImageReward",
    prompt_path='./prompt_files/image_rewards_benchmark.json',
    model_name="stable-diffusion-xl",
  )

fkd_args = dict(
    lmbda=2.0,
    num_particles=10,
    adaptive_resampling=True,
    resample_frequency=20,
    time_steps=100,
    potential_type='max',
    resampling_t_start=20,
    resampling_t_end=50,
    guidance_reward_fn='ImageReward',
    use_smc=True,
   )

args = argparse.Namespace(**args, **fkd_args)
args

In [ ]:
args.num_inference_steps = fkd_args["time_steps"]
fkd_args

In [ ]:
# seed everything
torch.manual_seed(args.seed)
torch.cuda.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)

In [ ]:
from fks_utils import get_model

pipeline = get_model(args.model_name)
# pipeline = pipeline.to("cuda")
pipeline.enable_model_cpu_offload()

In [ ]:
# set output directory
cur_time = datetime.now().strftime("%Y%m%d-%H%M%S")
output_dir = os.path.join(args.output_dir, cur_time)
os.makedirs(output_dir, exist_ok=False)
arg_path = os.path.join(output_dir, "args.json")
with open(arg_path, "w") as f:
    json.dump(vars(args), f, indent=4)

score_path = os.path.join(output_dir, "scores.jsonl")
images_path = os.path.join(output_dir, "images")
os.makedirs(images_path, exist_ok=False)

metrics_to_compute = args.metrics_to_compute.split("#")


# cache metric fns
do_eval(
    prompt=["test"],
    images=[Image.new("RGB", (224, 224))],
    metrics_to_compute=metrics_to_compute,
    )


In [ ]:
# add prompts for generation
prompt_data = [
    {"prompt": "a photo of a brown knife and a blue donut"},
    {"prompt": "a photo of a blue clock and a white cup"},
    {"prompt": "a photo of an orange cow and a purple sandwich"},
    {"prompt": "a photo of a yellow bird and a black motorcycle"},
    {"prompt": "a photo of a green tennis racket and a black dog"},
    {"prompt": "a green stop sign in a red field"},
]
len(prompt_data)

In [ ]:
### Timestep Visualization
import math
import numpy as np
import matplotlib.pyplot as plt
import torch

from fkd_diffusers.rewards import get_reward_function
if "xl" in args.model_name:
    from fkd_diffusers.fkd_pipeline_sdxl import latent_to_decode
else:
    from fkd_diffusers.fkd_pipeline_sd import latent_to_decode

show_best_particle = False
frame_stride = 5            # use 2/5/10 to reduce overhead
max_frames_to_show = 30     # cap display width for readability

with open(score_path, "w") as score_f:
    for prompt_idx, item in enumerate(prompt_data):
        torch.manual_seed(0)
        torch.cuda.manual_seed(0)
        torch.cuda.manual_seed_all(0)

        prompt = [item["prompt"]] * fkd_args["num_particles"]

        latent_vecs = [] # each item shape: [num_particles, latent_dim]
        denoise_frames = []
        denoise_timesteps = []

        def capture_step(pipe, step, t, callback_kwargs):
            if (step % frame_stride == 0) or (step == fkd_args["time_steps"] - 1):
                with torch.no_grad():
                    latents_step = callback_kwargs["latents"].detach()
                    # cosine similarity in latent space as correlation
                    v = latents_step.float().flatten(start_dim=1)
                    v = v / (v.norm(dim=1, keepdim=True) + 1e-8)
                    latent_vecs.append(v.cpu().numpy())

                    decoded = latent_to_decode(
                        model=pipe, output_type="pil", latents=latents_step
                    )
                    pil_imgs = pipe.image_processor.postprocess(decoded, output_type="pil")
                denoise_frames.append(list(pil_imgs))
                denoise_timesteps.append(int(t))
            return callback_kwargs

        start_time = datetime.now()
        images = pipeline(
            prompt,
            num_inference_steps=fkd_args["time_steps"],
            eta=args.eta,
            fkd_args=fkd_args,
            callback_on_step_end=capture_step,
            callback_on_step_end_tensor_inputs=["latents"],
        )[0]
        end_time = datetime.now()

        time_taken = end_time - start_time

        results = do_eval(prompt=prompt, images=images,             
                    metrics_to_compute=metrics_to_compute
        )
        guidance_reward = np.array(results["ImageReward"]["result"])
        sorted_idx = np.argsort(guidance_reward)[::-1]
        best_particle_idx_original = int(sorted_idx[0])

        images = [images[i] for i in sorted_idx]

        results["time_taken"] = time_taken.total_seconds()
        results["prompt"] = prompt
        results["prompt_index"] = prompt_idx

        image_fpath = os.path.join(images_path, f"{prompt_idx}.png")
        results["image_path"] = image_fpath
        score_f.write(json.dumps(results) + "\n")

        if len(denoise_frames) > max_frames_to_show:
            keep = np.linspace(0, len(denoise_frames) - 1, max_frames_to_show, dtype=int)
        else:
            keep = np.arange(len(denoise_frames))

        # Rewards for exactly the timestep images that will be displayed
        display_step_rewards = []
        for k in keep:
            # same particle order as visualization (sorted by final reward)
            step_imgs_ranked = [denoise_frames[k][i] for i in sorted_idx]
            step_rewards = get_reward_function(
                fkd_args["guidance_reward_fn"],
                images=step_imgs_ranked,
                prompts=prompt,
                metric_to_chase=fkd_args.get("metric_to_chase", None),
            )
            display_step_rewards.append(np.array(step_rewards, dtype=float))

        # Print reward table
        print(f"\nPrompt {prompt_idx}: {prompt[0]}")
        for col_idx, k in enumerate(keep):
            t_val = denoise_timesteps[k]
            vals = " | ".join(
                [f"p{p}:{display_step_rewards[col_idx][p]:.3f}" for p in range(len(sorted_idx))]
            )
            print(f"step={k:03d}, t={t_val:>4} -> {vals}")

        # Reorder per-step frames to match final ranking (best row first)
        reordered_frames = [[step_imgs[i] for i in sorted_idx] for step_imgs in denoise_frames]
        sorted_rewards = guidance_reward[sorted_idx]

        n_particles = args.num_particles
        pairwise_by_step = []
        pair_traj = {(i, j): [] for i in range(n_particles) for j in range(i + 1, n_particles)
                    }
        for k in keep:
            V = latent_vecs[k][sorted_idx]
            S = V @ V.T
            pairwise_by_step.append(S)
            for i in range(n_particles):
                for j in range(i + 1, n_particles):
                    pair_traj[(i, j)].append(S[i, j])

        # Print values for each timestep
        print("\nPairwise cosine at each displayed timestep")
        for c, k in enumerate(keep):
            t_val = denoise_timesteps[k]
            vals = []
            for i in range(n_particles):
                for j in range(i + 1, n_particles):
                    vals.append(f"p{i}-p{j}:{pairwise_by_step[c][i, j]:.3f}")
            print(f"step={k:03d}, t={t_val:>4} -> " + " | ".join(vals))

        if show_best_particle:
            traj_imgs = [reordered_frames[k][0] for k in keep]
            traj_ts = [denoise_timesteps[k] for k in keep]

            n_panels = len(traj_imgs) + 1
            cols = 8
            rows = math.ceil(n_panels / cols)

            fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.6))
            axes = np.array(axes).reshape(rows, cols)

            for ax in axes.flat:
                ax.axis("off")

            for i, (im, ts) in enumerate(zip(traj_imgs, traj_ts)):
                axes.flat[i].imshow(im)
                axes.flat[i].set_title(
                    f"t={ts}\nr={display_step_rewards[i][0]:.2f}", fontsize=8
                )

            axes.flat[len(traj_imgs)].imshow(images[0])
            axes.flat[len(traj_imgs)].set_title(
                f"final best r={sorted_rewards[0]:.2f}", fontsize=8
            )

            fig.suptitle(prompt[0], fontsize=12)
            fig.tight_layout()
            plt.savefig(image_fpath, dpi=180)
            plt.show()
            plt.close(fig)

        else:
            # Show all particles: rows=particles (sorted by final reward), cols=timesteps + final
            n_particles = len(sorted_idx)
            n_steps = len(keep)
            cols = n_steps + 1
            rows = n_particles

            fig, axes = plt.subplots(
                rows,
                cols,
                figsize=(max(12, cols * 1.8), max(6, rows * 2.2)),
                squeeze=False
            )

            for r in range(rows):
                for c, k in enumerate(keep):
                    ax = axes[r, c]
                    ax.imshow(reordered_frames[k][r])
                    ax.axis("off")
                    ax.set_title(
                        f"t={denoise_timesteps[k]}\nr={display_step_rewards[c][r]:.2f}", fontsize=8
                    )

                # Last column: final image of that particle
                ax_final = axes[r, -1]
                ax_final.imshow(images[r])
                ax_final.axis("off")
                ax_final.set_title(f"final r={sorted_rewards[r]:.2f}", fontsize=8)

            fig.suptitle(
                f"{prompt[0]}\nRows: all particles (sorted by final reward), Cols: timesteps + final",
                fontsize=11
            )
            fig.tight_layout()
            plt.savefig(image_fpath, dpi=180)
            plt.show()
            plt.close(fig)

        torch.cuda.empty_cache()